In [4]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
  Rolling back uninstall of pip
  Moving to c:\users\nishanth\appdata\roaming\python\python312\scripts\pip.exe
   from C:\Users\Nishanth\AppData\Local\Temp\pip-uninstall-gxihxqw9\pip.exe
  Moving to c:\users\nishanth\appdata\roaming\python\python312\scripts\pip3.12.exe
   from C:\Users\Nishanth\AppData\Local\Temp\pip-uninstall-gxihxqw9\pip3.12.exe
  Moving to c:\users\nishanth\appdata\roaming\python\python312\scripts\pip3.exe
   from C:\Users\Nishanth\AppData\Local\Temp\pip-uninstall-gxihxqw9\pip3.exe
  Moving to c:\users\nishanth\appdata\roaming\python\python312\site-packages\pip-24.3.1.dist-info\
   from C:\Users\Nishanth\AppData\Roaming\Python\Python312\site-packages\~ip-24.3.1.dist-info
  Moving to

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Python312\\Lib\\site-packages\\pip\\__init__.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install osmnx

  Using cached osmnx-2.0.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached osmnx-2.0.4-py3-none-any.whl (100 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.Collecting tqdm



ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python312\\Scripts\\tqdm.exe' -> 'c:\\Python312\\Scripts\\tqdm.exe.deleteme'


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import gzip
import pickle
import pandas as pd
import osmnx as ox
import networkx as nx
from tqdm import tqdm

In [10]:
CITY_NAME        = "Chicago, Illinois, USA"
INPUT_CSV        = "Taxi_Trips__2024-__20250611.csv"                # your raw file
OUTPUT_PKL_GZ    = "phase1_trajectories.pkl.gz"
CHUNK_ROWS       = 50_000                     # rows per batch
N_JOBS           = 8

In [11]:
print(f"Downloading/Loading road network for {CITY_NAME} …")
G = ox.graph_from_place(CITY_NAME, network_type="drive")
# (Optional) speeds & travel times if your OSMnx ≥1.1
try:
    G = ox.speed.add_edge_speeds(G)
    G = ox.speed.add_edge_travel_times(G)
except AttributeError:
    # fallback: give every edge a default 40 km/h
    for _, _, _, data in G.edges(keys=True, data=True):
        data["speed_kph"] = data.get("speed_kph", 40)
        data["travel_time"] = data["length"] / (data["speed_kph"] * 1000 / 3600)


Downloading/Loading road network for Chicago, Illinois, USA …


In [12]:
def nearest_nodes_batch(latitudes, longitudes):
    return ox.distance.nearest_nodes(G, X=longitudes, Y=latitudes, return_dist=False)


In [13]:
path_cache = {}
def shortest_path_edges(o_node, d_node):
    key = (o_node, d_node)
    if key in path_cache:            # cache hit
        return path_cache[key]
    try:                             # compute new path
        path_nodes = nx.shortest_path(G, o_node, d_node, weight="travel_time")
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        path_cache[key] = None
        return None
    # optional: store node list; edge IDs or tuples are OK for clustering
    path_cache[key] = path_nodes
    return path_nodes

In [19]:
trajectories = []
total_rows   = sum(1 for _ in open(INPUT_CSV)) - 1   # minus header
reader = pd.read_csv(
    INPUT_CSV,
    usecols=["Trip_ID", "Pickup_Centroid_Latitude", "Pickup_Centroid_Longitude",
             "Dropoff_Centroid_Latitude", "Dropoff_Centroid_Longitude", "Trip_Start_Timestamp"],
    chunksize=CHUNK_ROWS
)

processed = 0
for chunk in reader:
    # skip rows with missing geo
    chunk.dropna(subset=["Pickup_Centroid_Latitude","Pickup_Centroid_Longitude","Dropoff_Centroid_Latitude","Dropoff_Centroid_Longitude"],
                 inplace=True)

    # batch nearest‑node lookup
    chunk["o_node"] = nearest_nodes_batch(chunk["Pickup_Centroid_Latitude"].values,
                                          chunk["Pickup_Centroid_Longitude"].values)
    chunk["d_node"] = nearest_nodes_batch(chunk["Dropoff_Centroid_Latitude"].values,
                                          chunk["Dropoff_Centroid_Longitude"].values)

    # build trajectory list for the chunk
    for row in tqdm(chunk.itertuples(index=False),
                    total=len(chunk),
                    desc=f"Chunk {processed//CHUNK_ROWS + 1}"):
        path_nodes = shortest_path_edges(row.o_node, row.d_node)
        if path_nodes is None:
            continue  # skip trips with no path

        trajectories.append(
            dict(
                trip_id    = row.Trip_ID,
                path       = path_nodes,          # list[int] node ids
                timestamp  = row.Trip_Start_Timestamp        # original request time
            )
        )
    processed += len(chunk)
    print(f"  → processed {processed:,}/{total_rows:,} rows, "
          f"kept {len(trajectories):,} trajectories so far")

print(f"Finished scanning CSV. Total trajectories: {len(trajectories):,}")


Chunk 1: 100%|██████████| 45385/45385 [03:15<00:00, 232.46it/s] 


  → processed 45,385/563,814 rows, kept 43,114 trajectories so far


Chunk 1: 100%|██████████| 45640/45640 [01:30<00:00, 505.99it/s] 


  → processed 91,025/563,814 rows, kept 87,017 trajectories so far


Chunk 2: 100%|██████████| 45879/45879 [00:45<00:00, 1014.26it/s]


...

  → processed 454,635/563,814 rows, kept 439,859 trajectories so far


Chunk 10: 100%|██████████| 44693/44693 [00:14<00:00, 3093.73it/s]


  → processed 499,328/563,814 rows, kept 482,987 trajectories so far


Chunk 10: 100%|██████████| 12617/12617 [00:03<00:00, 3617.82it/s]

  → processed 511,945/563,814 rows, kept 495,304 trajectories so far
Finished scanning CSV. Total trajectories: 495,304


In [20]:
meta = dict(
    created           = pd.Timestamp.utcnow().isoformat(),
    city              = CITY_NAME,
    osmnx_version     = ox.__version__,
    total_trajectories= len(trajectories)
)

with gzip.open(OUTPUT_PKL_GZ, "wb") as f:
    pickle.dump({"meta": meta, "data": trajectories}, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n✅  Saved trajectories to {OUTPUT_PKL_GZ} (size: {os.path.getsize(OUTPUT_PKL_GZ)/1e6:.1f} MB)")



✅  Saved trajectories to phase1_trajectories.pkl.gz (size: 16.8 MB)
